In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
from unet_model import *
from unet_dataset import *

NB_ID = "40"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32

MODEL_PATH = MODELS_DIR / "lucidrf_unet_checkpoint_v4.pth"

METADATA_PATH = get_unet_path(STAGE_SCALED, TEST, "mixed_metadata", extension="csv")
X_TEST_PATH = get_unet_path(STAGE_FINAL, TEST, "X_test")
Y_TEST_PATH = get_unet_path(STAGE_FINAL, TEST, "Y_test")

print(f"Running on Device: {DEVICE}")

In [ ]:
print("\n[1/3] Loading Data...")
test_dataset = LucidRFDataset(X_TEST_PATH, Y_TEST_PATH)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
df = pd.read_csv(METADATA_PATH)
print(f"   Test Set Size: {len(test_dataset)} samples")

In [ ]:
print("\n[2/3] Loading Model...")
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = LucidRF_UNet(n_channels=2, n_classes=2).to(DEVICE)

# Load weights
if torch.cuda.is_available():
    state_dict = torch.load(MODEL_PATH)
else:
    state_dict = torch.load(MODEL_PATH, map_location=torch.device('cpu'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("   Weights loaded successfully.")

In [ ]:
print("\n[3/3] Running Evaluation Loop...")

mse_losses = []
sinr_inputs = []
sinr_outputs = []
criterion = nn.MSELoss(reduction='none')

# Helper: Compute Power of a batch (B, 2, L) -> (B,)
def batch_power(tensor):
    # Sum of squares across channels and time, averaged by length
    # Note: Using mean preserves the scale relative to 1.0
    return torch.mean(tensor**2, dim=(1, 2))

# Helper: Compute SINR in dB for a batch
def batch_sinr_db(signal, noise):
    p_sig = batch_power(signal)
    p_noise = batch_power(noise)
    # Add epsilon for numerical stability
    return 10 * torch.log10(p_sig / (p_noise + 1e-10))

with torch.no_grad():
    for x_batch, y_batch in tqdm(test_loader, desc="Evaluating"):
        x_batch = x_batch.to(DEVICE) # Noisy Input
        y_batch = y_batch.to(DEVICE) # Clean Target (Ground Truth)

        # Model Prediction
        y_pred = model(x_batch)

        # MSE Calculation
        loss_tensor = criterion(y_pred, y_batch)
        loss_per_sample = loss_tensor.mean(dim=(1, 2))
        mse_losses.extend(loss_per_sample.cpu().numpy())

        # SINR Calculation
        # Input Noise = Input - Target
        noise_in = x_batch - y_batch
        
        # Output Noise (Residual) = Prediction - Target
        noise_out = y_pred - y_batch

        # Calculate dB metrics
        s_in_db = batch_sinr_db(y_batch, noise_in)
        s_out_db = batch_sinr_db(y_batch, noise_out)

        sinr_inputs.extend(s_in_db.cpu().numpy())
        sinr_outputs.extend(s_out_db.cpu().numpy())

In [ ]:
avg_mse = np.mean(mse_losses)
sinr_inputs = np.array(sinr_inputs)
sinr_outputs = np.array(sinr_outputs)
sinr_gain = sinr_outputs - sinr_inputs

print("\n" + "="*40)
print("       EVALUATION REPORT")
print("="*40)
print(f"Global MSE Loss   : {avg_mse:.6f}")
print("-" * 40)
print(f"Avg Input SINR    : {np.mean(sinr_inputs):.2f} dB")
print(f"Avg Output SINR   : {np.mean(sinr_outputs):.2f} dB")
print(f"Avg SINR GAIN     : {np.mean(sinr_gain):.2f} dB")
print("="*40)

In [ ]:
print("Merging Metrics with Metadata...")

df['Input_SINR_dB'] = sinr_inputs
df['Output_SINR_dB'] = sinr_outputs
df['Model_Gain_dB'] = sinr_gain
# Note: mse_losses is batch-averaged in loop, might not perfectly align row-by-row if batch size varies, 
# but is fine for general distribution. For exact row matching, we'd calc MSE per sample.
df['MSE_Loss'] = mse_losses[:len(df)] 

print(df[['interference_type', 'Input_SINR_dB', 'Model_Gain_dB']].head())

In [ ]:
sns.set_theme(style="whitegrid")

# --- Plot A: Histogram of Gains ---
sns.histplot(df['Model_Gain_dB'], bins=50, kde=True, color='green', alpha=0.6)
plt.axvline(df['Model_Gain_dB'].mean(), color='red', linestyle='--', label=f"Mean: {df['Model_Gain_dB'].mean():.2f} dB")
plt.title('Distribution of SINR Improvement (Test Set)')
plt.xlabel('SINR Gain (dB)')
plt.legend()
save_plot("sinr_gain_histogram", machine_id="B", nb_id=NB_ID, fig_id="01")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.violinplot(data=df, x='interference_type', y='Model_Gain_dB', inner='quartile', palette="muted")
plt.title('Model Performance Distribution by Interference Type')
plt.ylabel('SINR Improvement (dB)')
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3)
save_plot("performance_violin_by_type", machine_id="B", nb_id=NB_ID, fig_id="02")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Input_SINR_dB', y='Model_Gain_dB', hue='interference_type', alpha=0.6, palette='deep')
plt.axhline(0, color='red', linestyle='--', label='Zero Gain (Identity)')
plt.title('Impact of Input Signal Quality on Model Performance')
plt.xlabel('Input SINR (dB) - "How noisy was it?"')
plt.ylabel('SINR Gain (dB) - "How much did we fix it?"')
plt.legend(title='Interference Type')
plt.grid(True, alpha=0.3)
save_plot("gain_vs_input_quality_scatter", machine_id="B", nb_id=NB_ID, fig_id="03")
plt.show()

In [ ]:
failures = df[df['Model_Gain_dB'] < -2.0]
if not failures.empty:
    worst_idx = failures.sort_values('Model_Gain_dB').index[0]
    print(f"\nVisualizing Worst Failure (Index {worst_idx}):")
    print(failures.loc[worst_idx])

    # Load specific sample
    x_fail, y_fail = test_dataset[worst_idx]
    x_fail = x_fail.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        y_pred_fail = model(x_fail)
    
    # Plot Spectrograms
    def plot_spectrogram(tensor, title):
        sig = tensor.squeeze().cpu().numpy()
        complex_sig = sig[0] + 1j * sig[1]
        plt.specgram(complex_sig, NFFT=1024, Fs=1, noverlap=512)
        plt.title(title)

    plt.figure(figsize=(15, 4))
    plt.subplot(1, 3, 1)
    plot_spectrogram(x_fail, f"Input (Noisy) - Idx {worst_idx}")
    plt.subplot(1, 3, 2)
    plot_spectrogram(y_pred_fail, f"Prediction (Gain: {df.loc[worst_idx, 'Model_Gain_dB']:.2f} dB)")
    plt.subplot(1, 3, 3)
    plot_spectrogram(y_fail.unsqueeze(0), "Ground Truth (Clean)")
    plt.tight_layout()
    save_plot(f"failure_analysis_idx{worst_idx}", machine_id="B", nb_id=NB_ID, fig_id="04")
    plt.show()
else:
    print("\nNo catastrophic failures (<-2 dB) found to visualize.")